In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
import seaborn as sns
import torch.nn as nn
import torch
import sys
import os

from data_pipeline import get_full_sequence_pipeline

CONFIG = {
    "base": dict(c1=32,c2=64,h=128,layers=2),
    "small": dict(c1=16,c2=32,h=64,layers=2),
    "tiny": dict(c1=8,c2=16,h=32,layers=1)
}

def run_cnnlstm(mode: int, size="base", window=None, batch_size=512):
  (loaders, scaler, le) = get_full_sequence_pipeline(mode, window=window, batch_size=batch_size)

  train_loader, val_loader, test_loader, n_features, n_classes = loaders


  class CNN_LSTM(nn.Module):
    def __init__(self, n_features, n_classes):
      super().__init__()

      cfg = CONFIG[size]

      self.cnn = nn.Sequential(
          nn.Conv1d(n_features, cfg['c1'], kernel_size=3, padding=1),
          nn.ReLU(),
          nn.Conv1d(cfg['c1'], cfg['c2'], kernel_size=3, padding=1),
          nn.ReLU(),
      )

      self.lstm = nn.LSTM(
          input_size=cfg['c2'],
          hidden_size=cfg['h'],
          num_layers=cfg['layers'],
          batch_first=True
      )

      self.dropout = nn.Dropout(0.3)

      self.attn = nn.Linear(cfg['h'], 1)

      self.fc = nn.Linear(cfg['h'], n_classes)

    def forward(self, x):
      x = x.permute(0, 2, 1)
      x = self.cnn(x)
      x = x.permute(0, 2, 1)
      lstm_out, _ = self.lstm(x)
      weights = torch.softmax(self.attn(lstm_out), dim=1)
      context = torch.sum(weights * lstm_out, dim=1)
      context = self.dropout(context)
      out = self.fc(context)
      return out

  criterion = nn.CrossEntropyLoss()
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = CNN_LSTM(n_features, n_classes).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

  def cnnlstm_epoch(loader, train=True):
    if train:
      model.train()
    else:
      model.eval()

    total_correct = 0
    total_loss = 0
    n = 0

    all_preds = []
    all_true = []

    with torch.set_grad_enabled(train):
      for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)

        if train:
          optimizer.zero_grad()
          loss.backward()
          torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
          optimizer.step()

        total_loss += loss.item() * y.shape[0]
        n += y.shape[0]

        preds = logits.argmax(dim=1).detach().cpu().numpy()
        total_correct += (preds == y.detach().cpu().numpy()).sum()
        all_preds.append(preds)
        all_true.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_true = np.concatenate(all_true)
    avg_loss = total_loss / n
    macro_f1 = f1_score(all_true, all_preds, average='macro')
    acc = accuracy_score(all_true, all_preds)

    return{
        'loss': avg_loss,
        'f1': macro_f1,
        'acc': acc,
        'preds': all_preds,
        'true': all_true
    }

  train_losses = []
  val_losses = []
  train_f1s = []
  val_f1s = []
  train_accs = []
  val_accs = []

  for epoch in range(10):
    train_epoch = cnnlstm_epoch(train_loader)
    train_loss = train_epoch['loss']
    train_f1 = train_epoch['f1']
    train_acc = train_epoch['acc']
    val_epoch = cnnlstm_epoch(val_loader, train=False)
    val_loss = val_epoch['loss']
    val_f1 = val_epoch['f1']
    val_acc = val_epoch['acc']
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

  test_epoch = cnnlstm_epoch(test_loader, train=False)
  test_loss = test_epoch['loss']
  test_f1 = test_epoch['f1']
  test_preds = test_epoch['preds']
  test_trues = test_epoch['true']

  accuracy = accuracy_score(test_trues, test_preds)
  precision, recall, f1, _ = precision_recall_fscore_support(test_trues, test_preds, average='macro', zero_division=0)
  cm = confusion_matrix(test_trues, test_preds)

  return {
      'accuracy': accuracy,
      'precision': precision,
      'recall': recall,
      'f1': f1,
      'train_loss': train_losses,
      'val_loss': val_losses,
      'confusion_matrix': cm,
      'history': val_losses
  }

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files/data_pipeline.py:202: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files/data_pipeline.py:202: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files/data_pipeline.py:202: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/content/drive/MyDrive

Epoch 0 - Train Loss: 0.6076, Train F1: 0.0616, Train Acc: 0.8524, Val Loss: 0.5830, Val F1: 0.0614, Val Acc: 0.8546
Epoch 1 - Train Loss: 0.3272, Train F1: 0.1694, Train Acc: 0.9146, Val Loss: 0.0723, Val F1: 0.3054, Val Acc: 0.9822
Epoch 2 - Train Loss: 0.0517, Train F1: 0.4759, Train Acc: 0.9865, Val Loss: 0.0406, Val F1: 0.5195, Val Acc: 0.9885
Epoch 3 - Train Loss: 0.0313, Train F1: 0.5755, Train Acc: 0.9924, Val Loss: 0.0250, Val F1: 0.5930, Val Acc: 0.9939
Epoch 4 - Train Loss: 0.0201, Train F1: 0.6287, Train Acc: 0.9951, Val Loss: 0.0226, Val F1: 0.6327, Val Acc: 0.9944
Epoch 5 - Train Loss: 0.0177, Train F1: 0.6466, Train Acc: 0.9955, Val Loss: 0.0152, Val F1: 0.6591, Val Acc: 0.9962
Epoch 6 - Train Loss: 0.0145, Train F1: 0.6603, Train Acc: 0.9963, Val Loss: 0.0140, Val F1: 0.6694, Val Acc: 0.9966
Epoch 7 - Train Loss: 0.0160, Train F1: 0.6582, Train Acc: 0.9959, Val Loss: 0.0136, Val F1: 0.6661, Val Acc: 0.9965
Epoch 8 - Train Loss: 0.0121, Train F1: 0.6706, Train Acc: 0.996